# Query Snowflake-Managed Iceberg Tables from Apache Spark

This notebook connects Spark 4.0 to Snowflake-managed Iceberg tables through Horizon Catalog.
No data is copied — Spark reads the open Iceberg files directly.

**Prerequisites:**
- Conda environment: `conda activate fleet-spark`
- Packages: `pip install pyspark==4.0.0 jupyter python-dotenv`

In [1]:
import os
from dotenv import load_dotenv
load_dotenv('config.env')

# Configuration
# Hostnames can't contain '_' (fails TLS hostname validation), so use the hyphenated URL form
SNOWFLAKE_ACCOUNT_URL = os.getenv('SNOWFLAKE_ACCOUNT', '').replace('.', '-').replace('_', '-').lower()
SNOWFLAKE_PAT = os.getenv('SNOWFLAKE_PAT', '')
SNOWFLAKE_DATABASE = os.getenv('SNOWFLAKE_DATABASE', 'FLEET_DB')
SNOWFLAKE_ROLE = os.getenv('SNOWFLAKE_ROLE', 'ACCOUNTADMIN')

ICEBERG_VERSION = '1.10.1'
SCALA_VERSION = '2.13'

print(f'Account: {SNOWFLAKE_ACCOUNT_URL}')
print(f'Database: {SNOWFLAKE_DATABASE}')

Account: sfseapac-abarron-aws1
Database: FLEET_DB


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Fleet Analytics - Iceberg Interop') \
    .master('local[*]') \
    .config('spark.driver.host', '127.0.0.1') \
    .config('spark.driver.bindAddress', '127.0.0.1') \
    .config('spark.jars.packages',
            f'org.apache.iceberg:iceberg-spark-runtime-4.0_{SCALA_VERSION}:{ICEBERG_VERSION},'
            f'org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION}') \
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions') \
    .config('spark.sql.catalog.horizon', 'org.apache.iceberg.spark.SparkCatalog') \
    .config('spark.sql.catalog.horizon.type', 'rest') \
    .config('spark.sql.catalog.horizon.uri',
            f'https://{SNOWFLAKE_ACCOUNT_URL}.snowflakecomputing.com/polaris/api/catalog') \
    .config('spark.sql.catalog.horizon.oauth2-server-uri',
            f'https://{SNOWFLAKE_ACCOUNT_URL}.snowflakecomputing.com/polaris/api/catalog/v1/oauth/tokens') \
    .config('spark.sql.catalog.horizon.credential', SNOWFLAKE_PAT) \
    .config('spark.sql.catalog.horizon.warehouse', SNOWFLAKE_DATABASE) \
    .config('spark.sql.catalog.horizon.scope', f'session:role:{SNOWFLAKE_ROLE}') \
    .config('spark.sql.catalog.horizon.header.X-Iceberg-Access-Delegation',
            'vended-credentials') \
    .config('spark.sql.iceberg.vectorization.enabled', 'false') \
    .getOrCreate()

print(f'Spark {spark.version} connected to Horizon Catalog!')

:: loading settings :: url = jar:file:/opt/anaconda3/envs/fleet-spark/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/abarron/.ivy2.5.2/cache
The jars for the packages stored in: /Users/abarron/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9fb4c8cf-215a-4509-8520-0ea9d524525e;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.10.1 in central
:: resolution report :: resolve 77ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.iceberg#iceberg-aws-bundle;1.10.1 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 from central in [default]
	---------------------------------------------------------------------
	|   

Spark 4.0.0 connected to Horizon Catalog!


## List Tables

All Iceberg tables (including Dynamic Iceberg Tables) are visible through Horizon Catalog.

In [3]:
spark.sql('SHOW TABLES IN horizon.RAW').show(truncate=False)
spark.sql('SHOW TABLES IN horizon.ANALYTICS').show(truncate=False)

26/10/26 08:43:58 WARN AuthManagers: Inferring rest.auth.type=oauth2 since property credential was provided. Please explicitly set rest.auth.type to avoid this warning.


+---------+------------------------+-----------+
|namespace|tableName               |isTemporary|
+---------+------------------------+-----------+
|RAW      |MAINTENANCE_LOGS        |false      |
|RAW      |SENSOR_READINGS         |false      |
|RAW      |VEHICLE_REGISTRY        |false      |
|RAW      |VEHICLE_TELEMETRY_STREAM|false      |
+---------+------------------------+-----------+

+---------+-------------------+-----------+
|namespace|tableName          |isTemporary|
+---------+-------------------+-----------+
|ANALYTICS|DAILY_FLEET_SUMMARY|false      |
|ANALYTICS|TELEMETRY_ENRICHED |false      |
+---------+-------------------+-----------+



## Query Structured Data

Read the vehicle registry — standard columnar data.

In [4]:
spark.sql("""
    SELECT VEHICLE_ID, MAKE, MODEL, YEAR, FLEET_REGION
    FROM horizon.RAW.VEHICLE_REGISTRY
    LIMIT 10
""").show()

[Stage 0:>                                                          (0 + 1) / 1]

+----------+------------+---------+----+-----------------+
|VEHICLE_ID|        MAKE|    MODEL|YEAR|     FLEET_REGION|
+----------+------------+---------+----+-----------------+
|   VH-0003|         Ram|ProMaster|2023|          Midwest|
|   VH-0011|   Chevrolet|  Express|2025|       California|
|   VH-0019|Freightliner| Cascadia|2021|        Northeast|
|   VH-0027|      Toyota|   Tacoma|2023|    Mountain West|
|   VH-0035|        Ford|  Transit|2025|Pacific Northwest|
|   VH-0043|         Ram|ProMaster|2021|          Midwest|
|   VH-0051|   Chevrolet|  Express|2023|       California|
|   VH-0059|Freightliner| Cascadia|2025|        Northeast|
|   VH-0067|      Toyota|   Tacoma|2021|    Mountain West|
|   VH-0075|        Ford|  Transit|2023|Pacific Northwest|
+----------+------------+---------+----+-----------------+



## Query VARIANT Data with `variant_get()`

Spark 4.0 supports the Iceberg V3 VARIANT type natively. Use `variant_get()` to extract nested fields from JSON telemetry.

In [5]:
df = spark.sql("""
    SELECT
        VEHICLE_ID,
        EVENT_TIMESTAMP,
        variant_get(TELEMETRY_DATA, '$.speed_mph', 'float') AS speed_mph,
        variant_get(TELEMETRY_DATA, '$.engine.temperature_f', 'int') AS engine_temp,
        variant_get(TELEMETRY_DATA, '$.location.lat', 'float') AS latitude,
        variant_get(TELEMETRY_DATA, '$.location.lon', 'float') AS longitude
    FROM horizon.RAW.VEHICLE_TELEMETRY_STREAM
    WHERE variant_get(TELEMETRY_DATA, '$.speed_mph', 'float') > 60
    LIMIT 10
""")
df.show()

26/10/26 08:44:34 WARN CorruptStatistics: Ignoring statistics because created_by is null or empty! See PARQUET-251 and PARQUET-297
[Stage 4:==================================================>      (15 + 2) / 17]

+----------+--------------------+---------+-----------+---------+-----------+
|VEHICLE_ID|     EVENT_TIMESTAMP|speed_mph|engine_temp| latitude|  longitude|
+----------+--------------------+---------+-----------+---------+-----------+
|   VH-0013|2026-10-25 13:51:...|     65.0|        202|41.459034|  -87.47726|
|   VH-0042|2026-10-25 13:55:...|     83.0|        214| 40.05104|-105.168304|
|   VH-0022|2026-10-25 13:53:...|     71.4|        193|40.058495| -105.07803|
|   VH-0022|2026-10-25 13:56:...|     85.9|        203|40.058594|  -105.0791|
|   VH-0019|2026-10-25 13:53:...|     62.6|        183|41.014637|  -74.18847|
|   VH-0022|2026-10-25 13:53:...|     68.4|        196|40.059006| -105.07793|
|   VH-0042|2026-10-25 13:54:...|     78.6|        210|40.050426| -105.16796|
|   VH-0042|2026-10-25 13:51:...|     72.0|        210| 40.04924| -105.16656|
|   VH-0042|2026-10-25 13:53:...|     62.5|        211|40.049477|-105.167625|
|   VH-0013|2026-10-25 13:53:...|     72.5|        204|41.459743

## Query Dynamic Iceberg Tables

Dynamic Iceberg Tables are just regular Iceberg tables to external engines.

In [6]:
spark.sql("""
    SELECT *
    FROM horizon.ANALYTICS.DAILY_FLEET_SUMMARY
    ORDER BY summary_date DESC
    LIMIT 10
""").show()

+------------+------------+------------+---------------+-------------+-----------------+------------------+---------------+-------------------+
|SUMMARY_DATE|FLEET_REGION|TOTAL_EVENTS|ACTIVE_VEHICLES|AVG_SPEED_MPH|AVG_ENGINE_TEMP_F|AVG_FUEL_LEVEL_PCT|CRITICAL_EVENTS|CHECK_ENGINE_EVENTS|
+------------+------------+------------+---------------+-------------+-----------------+------------------+---------------+-------------------+
+------------+------------+------------+---------------+-------------+-----------------+------------------+---------------+-------------------+



## Aggregate Sensor Readings

Run analytics on the sensor readings table.

In [7]:
spark.sql("""
    SELECT
        VEHICLE_ID,
        COUNT(*) AS reading_count,
        ROUND(AVG(ENGINE_TEMP_F), 1) AS avg_engine_temp,
        ROUND(AVG(FUEL_CONSUMPTION_GPH), 2) AS avg_fuel_gph
    FROM horizon.RAW.SENSOR_READINGS
    GROUP BY VEHICLE_ID
    ORDER BY avg_fuel_gph DESC
    LIMIT 10
""").show()

[Stage 5:>                                                          (0 + 1) / 1]

+----------+-------------+---------------+------------+
|VEHICLE_ID|reading_count|avg_engine_temp|avg_fuel_gph|
+----------+-------------+---------------+------------+
|   VH-0015|          100|          179.6|       13.83|
|   VH-0006|          100|          161.9|        11.6|
|   VH-0021|          100|          145.0|       10.58|
|   VH-0041|          100|          115.2|        9.59|
|   VH-0047|          100|          170.7|        9.56|
|   VH-0014|          100|          154.7|        9.48|
|   VH-0083|          100|          249.4|        8.95|
|   VH-0093|          100|          178.4|        8.73|
|   VH-0070|          100|          217.9|         8.4|
|   VH-0080|          100|          147.9|        8.38|
+----------+-------------+---------------+------------+



26/10/26 08:45:40 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /private/var/folders/f_/09b02xrn531d1kc1hrz1_bj80000gn/T/spark-baa19977-d865-4083-be1e-6a8c4657137f/pyspark-a32dbb7b-78dd-4c86-a030-5342170ab01a. Falling back to Java IO way
java.io.IOException: Failed to delete: /private/var/folders/f_/09b02xrn531d1kc1hrz1_bj80000gn/T/spark-baa19977-d865-4083-be1e-6a8c4657137f/pyspark-a32dbb7b-78dd-4c86-a030-5342170ab01a
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:199)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:116)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1048)
	at org.apache.spark.util.ShutdownHookManag

In [ ]:
spark.stop()
print('Done! Spark read Snowflake-managed Iceberg tables with zero data movement.')